In [10]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..")))

from langchain_ollama import ChatOllama

from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda

from langchain_core.output_parsers import StrOutputParser

from app.prompts import RAG_PROMPT, RAG_PROMPT_WITH_HISTORY
from app.vectordb import load_vector_db
from app.retriever import get_retriever

In [2]:
vector_db = load_vector_db()

retriever = get_retriever(vector_db)

llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [3]:
def format_docs(docs):

    return "\n\n".join(

        f"Source: {doc.metadata.get('source')}, "
        f"Page: {doc.metadata.get('page', 'N/A')}\n"
        f"{doc.page_content}"

        for doc in docs
    )

In [4]:
docs = retriever.invoke("What is diabetes?")

print(format_docs(docs)[:800])

Source: ..\data\pdf\medical_handbook.pdf, Page: 2
2.2  DIABETES  MELLITUS  -  SAFE  COLD  MEDICATIONS  ----------------------------------------------  RISK  ASSESSMENT:  MODERATE  -  Focus  on  blood  glucose  monitoring  during  illness   SAFE  OPTIONS  (First  Choice):  -  Acetaminophen:  NO  effect  on  blood  glucose.  Safe.  -  Guaifenesin:  NO  effect  on  glucose.  Safe.  -  Dextromethorphan:  NO  effect  on  glucose.  Safe.  -  Loratadine/Cetirizine/Fexofenadine:  No  glucose  effects.  Safe.  -  Nasal  saline  spray:  Safe.  -  Oxymetazoline  nasal:  Safe.  No  systemic  effects  on  glucose.   USE  WITH  CAUTION:  -  Pseudoephedrine:  May  cause  mild  hyperglycemia.  Monitor  glucose.  -  Phenylephrine:  Similar  caution  as  pseudoephedrine.   CONTRAINDICATED/AVOID:  -  Oral  c


In [13]:
chain1 = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

In [14]:
response = chain1.invoke(
    "What is heart disease?"
)

print(response)

 I couldn't find information about heart disease in the provided context as it does not pertain to any of the patient records. Heart disease typically refers to a range of conditions that affect your heart, such as coronary artery disease (CAD), heart failure, and arrhythmias. It is important to note that some patients have a history of hypertension, which can be a risk factor for heart disease, but none of the provided records indicate a specific diagnosis of heart disease.


In [15]:
from app.chains import build_chain

chain1 = build_chain()

print(chain1.invoke("What is heart disease?"))

 I couldn't find information about heart disease in the provided context as it does not pertain to any of the patient records. Heart disease typically refers to a range of conditions that affect your heart, such as coronary artery disease (CAD), heart failure, and arrhythmias. It is important to note that some patients have a history of hypertension, which can be a risk factor for heart disease, but none of the provided records indicate a specific diagnosis of heart disease.


In [16]:
from langchain_core.runnables.history import RunnableWithMessageHistory

from app.memory import get_session_history

In [17]:
chain2 = (
    {
        "context": (
            RunnableLambda(lambda x: x["question"])
            | retriever
            | RunnableLambda(format_docs)
        ),
        "question": RunnableLambda(lambda x: x["question"]),
        "history": RunnableLambda(lambda x: x.get("history", [])),
    }
    | RAG_PROMPT_WITH_HISTORY
    | llm
    | StrOutputParser()
)

In [18]:
chatbot = RunnableWithMessageHistory(

    chain2,

    get_session_history,

    input_messages_key="question",

    history_messages_key="history"
)

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [19]:
response = chatbot.invoke(
    {
        "question": "What is heart disease?"
    },
    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)

print(response)

 I couldn't find information about heart disease in the provided context as it does not pertain to any of the patient records. Heart disease typically refers to a range of conditions that affect your heart, such as coronary artery disease (CAD), heart failure, and arrhythmias. It is important to note that some patients have a history of hypertension, which can be a risk factor for heart disease, but none of the provided records indicate a specific diagnosis of heart disease.


In [20]:
response = chatbot.invoke(
    {
        "question": "What is diabetes?"
    },
    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)

print(response)

 Diabetes Mellitus is a medical condition characterized by high levels of glucose (sugar) in the blood due to problems with insulin production or function. It is mentioned as one of the medical histories for both Timothy Walker and Robert Williams in the provided patient records.


In [21]:
response = chatbot.invoke(
    {
        "question": "Medicines for it?"
    },
    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)

print(response)

 The provided context does not specify the condition that requires medicines. However, based on the information in the documents, some medications mentioned include decongestants (such as Pseudoephedrine and Oxymetazoline), antihistamines (first-generation like Diphenhydramine and second-generation like Loratadine and Cetirizine), cough suppressants (like Dextromethorphan), expectorants (such as Guaifenesin), mucolytics (Acetylcysteine), throat lozenges, and combination products for multi-symptom relief.


In [22]:
response = chatbot.invoke(
    {
        "question": "Medicine for it?"
    },
    config={
        "configurable": {
            "session_id": "user2"
        }
    }
)

print(response)

 Based on the provided context, it seems that the question is asking about medication for a cold or cold symptoms. According to the information on Page 6 of the medical handbook, "Other cold medications generally safe." However, there are some interactions to be aware of, such as with Theophylline and certain decongestants like Pseudoephedrine (Page 6).

On Page 14, it is mentioned that non-sedating antihistamines like Cetirizine are safe for fever/body ache. On Page 10, it is stated that second-generation antihistamines like Loratadine and Cetirizine are safe and do not cause additive sedation.

On Page 11, it is mentioned that other cold medications generally safe. However, on Page 14, it is also noted that Oxymetazoline nasal (a decongestant) should be used with caution for no more than 3 days due to potential complex interactions.

Therefore, it would be best to consult a healthcare professional or refer to the specific medication guide for advice on appropriate cold medications, e